# Factors Affecting Output of Firms in Vietnam's Furniture Industry
## Kinh Tế Lượng — Panel Data Econometric Analysis

**Model:**
$$\ln(Output_{it}) = \beta_0 + \beta_1\ln(Labor_{it}) + \beta_2\ln(Capital_{it}) + \beta_3 Leverage_{it} + \beta_4\ln(Wage_{it}) + \beta_5\ln(Size_{it}) + \varepsilon_{it}$$

**Data:** GSO Enterprise Survey, Vietnam (2012–2018), VSIC Industry Code 31 (Furniture Manufacturing)

**Methods:** Pooled OLS · Fixed Effects (FE) · Random Effects (RE) · Hausman Test

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import statsmodels.api as sm
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS as LMPooledOLS
import warnings
import os
warnings.filterwarnings('ignore')

BASE_DIR = os.path.dirname(os.getcwd())
DATA_IN  = os.path.join(BASE_DIR, 'output', 'data', 'furniture_cleaned.csv')
FIG_DIR  = os.path.join(BASE_DIR, 'output', 'figures')
TBL_DIR  = os.path.join(BASE_DIR, 'output', 'tables')

df = pd.read_csv(DATA_IN)
print(f'Dataset: {df.shape[0]:,} observations, {df["firm_id"].nunique():,} unique firms, {df["year"].nunique()} years')
df.head()

## 1. Data Overview

In [ ]:
# Load pre-computed tables
desc_raw = pd.read_csv(os.path.join(TBL_DIR, 'table1_descriptive_raw.csv'), index_col=0)
desc_log = pd.read_csv(os.path.join(TBL_DIR, 'table2_descriptive_log.csv'), index_col=0)

print('=== Table 1: Descriptive Statistics (Raw Variables) ===')
display(desc_raw)
print()
print('=== Table 2: Descriptive Statistics (Log-Transformed Variables) ===')
display(desc_log)

In [ ]:
yr_tbl = pd.read_csv(os.path.join(TBL_DIR, 'table3_panel_by_year.csv'), index_col='year')
print('=== Table 3: Panel Structure by Year ===')
display(yr_tbl)

In [ ]:
# Figure 1: Distributions
img = mpimg.imread(os.path.join(FIG_DIR, 'fig1_distributions.png'))
plt.figure(figsize=(14, 8))
plt.imshow(img)
plt.axis('off')
plt.tight_layout()
plt.show()

## 2. Correlation Analysis

In [ ]:
corr = pd.read_csv(os.path.join(TBL_DIR, 'table5_correlation.csv'), index_col=0)
print('=== Table 5: Pearson Correlation Matrix ===')
display(corr.style.background_gradient(cmap='RdBu_r', vmin=-1, vmax=1).format('{:.3f}'))

In [ ]:
img = mpimg.imread(os.path.join(FIG_DIR, 'fig2_correlation_heatmap.png'))
plt.figure(figsize=(8, 7))
plt.imshow(img)
plt.axis('off')
plt.show()

## 3. Trends Over Time

In [ ]:
img = mpimg.imread(os.path.join(FIG_DIR, 'fig3_trends_over_time.png'))
plt.figure(figsize=(14, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

## 4. Scatter Plots: ln(Output) vs Predictors

In [ ]:
img = mpimg.imread(os.path.join(FIG_DIR, 'fig4_scatter_plots.png'))
plt.figure(figsize=(16, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

## 5. Regression Models

We estimate the following models:
1. **Pooled OLS** — baseline, ignores firm heterogeneity
2. **Fixed Effects (1-way)** — controls for firm-level unobserved heterogeneity
3. **Fixed Effects (2-way)** — controls for firm AND year effects
4. **Random Effects** — GLS with assumed random firm-level effects
5. **Hausman Test** — to choose between FE and RE

All models use **heteroscedasticity-robust standard errors**.

In [ ]:
DEPVAR  = 'lnOutput'
INDVARS = ['lnLabor', 'lnCapital', 'Leverage', 'lnWage', 'lnSize']
ALL_VARS = [DEPVAR] + INDVARS

df_model = df[['firm_id', 'year'] + ALL_VARS].dropna()
panel    = df_model.set_index(['firm_id', 'year']).sort_index()
Y        = panel[DEPVAR]
X_vars   = panel[INDVARS]
X_const  = sm.add_constant(X_vars)

ols_res = LMPooledOLS(Y, X_const).fit(cov_type='robust')
fe1_res = PanelOLS(Y, X_vars, entity_effects=True, time_effects=False).fit(cov_type='robust')
fe2_res = PanelOLS(Y, X_vars, entity_effects=True, time_effects=True).fit(cov_type='robust')
re_res  = RandomEffects(Y, X_const).fit(cov_type='robust')

print('Models estimated.')

In [ ]:
# Compile results table
results = pd.read_csv(os.path.join(TBL_DIR, 'table6_regression_results.csv'))
print('=== Table 6: Regression Results ===')
display(results)

In [ ]:
# Coefficient plot
img = mpimg.imread(os.path.join(FIG_DIR, 'fig6_coefficient_plot.png'))
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.axis('off')
plt.show()

## 6. Hausman Test

In [ ]:
hausman = pd.read_csv(os.path.join(TBL_DIR, 'table7_hausman_test.csv'))
display(hausman)

p = hausman['p-value'].values[0]
if p < 0.05:
    print(f'Hausman test: p = {p:.4f} < 0.05 -> Reject H0 -> Fixed Effects is preferred')
else:
    print(f'Hausman test: p = {p:.4f} >= 0.05 -> Fail to reject H0 -> Random Effects is preferred')

## 7. Diagnostics

In [ ]:
vif = pd.read_csv(os.path.join(TBL_DIR, 'diag_vif.csv'))
diag = pd.read_csv(os.path.join(TBL_DIR, 'table8_diagnostics.csv'))

print('=== VIF (Multicollinearity) ===')
display(vif)
print()
print('=== Other Diagnostics ===')
display(diag)

In [ ]:
img = mpimg.imread(os.path.join(FIG_DIR, 'fig7_residual_diagnostics.png'))
plt.figure(figsize=(14, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

## 8. Robustness: ln(VA) as Alternative Output Measure

In [ ]:
df_rob = df[['firm_id', 'year', 'lnVA'] + INDVARS].dropna()
df_rob = df_rob[np.isfinite(df_rob['lnVA'])]
panel_rob = df_rob.set_index(['firm_id', 'year']).sort_index()

fe_rob = PanelOLS(panel_rob['lnVA'], panel_rob[INDVARS],
                  entity_effects=True, time_effects=True).fit(cov_type='robust')
print('=== Robustness Check: FE (2-way) with ln(Value Added) ===')
print(fe_rob.summary)

## 9. Year Fixed Effects

In [ ]:
img = mpimg.imread(os.path.join(FIG_DIR, 'fig9_year_fixed_effects.png'))
plt.figure(figsize=(8, 5))
plt.imshow(img)
plt.axis('off')
plt.show()

## 10. Summary of Findings

| Variable | FE (2-way) Coeff | Significance | Interpretation |
|---|---|---|---|
| ln(Labor) | 0.484 | *** | 1% increase in labor -> 0.48% increase in output |
| ln(Capital) | 0.112 | ** | 1% increase in fixed assets -> 0.11% increase in output |
| Leverage | -0.027 | n.s. | Leverage ratio has no significant within-firm effect |
| ln(Wage) | 0.356 | *** | Higher average wages -> higher output (quality/skill effect) |
| ln(Size) | 0.257 | *** | Larger firms produce more even after controlling for other factors |

**Conclusion:** The Fixed Effects model (confirmed by Hausman test, p < 0.001) is the preferred specification. Labor and wage have the strongest effects on firm output in Vietnam's furniture industry.